<a href="https://colab.research.google.com/github/MuhamedZepcanin/MuhamedZepcanin.github.io/blob/main/upgraded_midterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# vehicles.py

class Vehicle:
    def __init__(self, vehicle_id, speed=50):
        self.vehicle_id = vehicle_id
        self.battery = 100
        self.status = "Idle"
        self.speed = speed

    def move(self, distance):
        raise NotImplementedError("Subclasses must implement move()")

    def charge(self, amount):
        self.battery = min(100, self.battery + amount)
        self.status = "Charging"


class GroundVehicle(Vehicle):
    def __init__(self, vehicle_id, terrain="road"):
        super().__init__(vehicle_id)
        self.terrain = terrain

    def move(self, distance):
        consumption = distance * 0.8
        if self.battery >= consumption:
            self.battery -= consumption
            self.status = f"Moving on {self.terrain}"
        else:
            self.status = "Low Battery"


class Drone(Vehicle):
    def move(self, distance):
        consumption = distance * 1.5
        if self.battery < 15:
            self.status = "Battery too low to fly"
        elif self.battery >= consumption:
            self.battery -= consumption
            self.status = "Flying"
        else:
            self.status = "Low Battery"


class UGV(Vehicle):
    def move(self, distance):
        consumption = distance
        if self.battery >= consumption:
            self.battery -= consumption
            self.status = "Moving (UGV)"
        else:
            self.status = "Low Battery"

    def deliver(self, weight):
        consumption = weight * 2
        if self.battery >= consumption:
            self.battery -= consumption
            self.status = f"Delivering {weight}kg"
        else:
            self.status = "Not enough battery for delivery"

In [ ]:
import streamlit as st
import random
import pandas as pd
import time
from vehicles import GroundVehicle, Drone, UGV

st.set_page_config(page_title="Tesla Fleet AI", layout="wide")

# -----------------------------
# 🎨 TESLA DARK THEME
# -----------------------------
st.markdown("""
<style>
body {
    background-color: #0b0c10;
    color: white;
}
.metric-card {
    background: #111217;
    padding: 20px;
    border-radius: 15px;
    text-align: center;
    box-shadow: 0px 0px 15px rgba(255,255,255,0.05);
}
.big-text {
    font-size: 32px;
    font-weight: bold;
}
.small-text {
    color: gray;
}
</style>
""", unsafe_allow_html=True)

st.title("🚗 Tesla-Style Fleet Dashboard")

# -----------------------------
# SESSION STATE
# -----------------------------
if "fleet" not in st.session_state:
    st.session_state.fleet = {}

if "history" not in st.session_state:
    st.session_state.history = []

fleet = st.session_state.fleet

# -----------------------------
# SIDEBAR (TESLA CONTROL PANEL)
# -----------------------------
st.sidebar.title("⚙️ Control Center")

vehicle_id = st.sidebar.text_input("Vehicle ID")
vehicle_type = st.sidebar.selectbox("Type", ["GroundVehicle", "Drone", "UGV"])
terrain = st.sidebar.selectbox("Terrain", ["road", "offroad"])

if st.sidebar.button("Add Vehicle"):
    if vehicle_id and vehicle_id not in fleet:
        if vehicle_type == "GroundVehicle":
            fleet[vehicle_id] = GroundVehicle(vehicle_id, terrain)
        elif vehicle_type == "Drone":
            fleet[vehicle_id] = Drone(vehicle_id)
        else:
            fleet[vehicle_id] = UGV(vehicle_id)
        st.sidebar.success("Vehicle Added")
    else:
        st.sidebar.error("Invalid ID")

# -----------------------------
# MAIN DASHBOARD
# -----------------------------
if fleet:
    selected = st.selectbox("Select Vehicle", list(fleet.keys()))
    v = fleet[selected]

    col1, col2, col3 = st.columns(3)

    # SPEED
    with col1:
        st.markdown(f"""
        <div class="metric-card">
            <div class="small-text">Speed</div>
            <div class="big-text">{v.speed} km/h</div>
        </div>
        """, unsafe_allow_html=True)

    # BATTERY
    with col2:
        st.markdown(f"""
        <div class="metric-card">
            <div class="small-text">Battery</div>
            <div class="big-text">{round(v.battery,1)}%</div>
        </div>
        """, unsafe_allow_html=True)

    # STATUS
    with col3:
        st.markdown(f"""
        <div class="metric-card">
            <div class="small-text">Status</div>
            <div class="big-text">{v.status}</div>
        </div>
        """, unsafe_allow_html=True)

    # -----------------------------
    # CONTROLS (TESLA BUTTONS)
    # -----------------------------
    st.markdown("## Controls")

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        if st.button("Drive"):
            d = random.randint(1, 10)
            v.move(d)
            st.session_state.history.append((selected, v.battery))

    with c2:
        if st.button("Charge"):
            c = random.randint(10, 30)
            v.charge(c)
            st.session_state.history.append((selected, v.battery))

    with c3:
        if isinstance(v, UGV):
            if st.button("Deliver"):
                w = random.randint(1, 10)
                v.deliver(w)
                st.session_state.history.append((selected, v.battery))

    with c4:
        if st.button("🤖 Autopilot"):
            if v.battery < 30:
                v.charge(30)
                action = "Charging"
            else:
                v.move(5)
                action = "Cruising"

            st.session_state.history.append((selected, v.battery))
            st.success(f"Autopilot: {action}")

    # -----------------------------
    # 📊 BATTERY GRAPH
    # -----------------------------
    st.markdown("## Energy Usage")

    if st.session_state.history:
        df = pd.DataFrame(st.session_state.history, columns=["Vehicle", "Battery"])
        st.line_chart(df.pivot(columns="Vehicle", values="Battery"))

    # -----------------------------
    # 🗺️ MAP (TESLA STYLE SIM)
    # -----------------------------
    st.markdown("## Live Location")

    map_data = []
    for vid in fleet:
        lat = 37.77 + random.uniform(-0.01, 0.01)
        lon = -122.41 + random.uniform(-0.01, 0.01)
        map_data.append([lat, lon])

    st.map(pd.DataFrame(map_data, columns=["lat", "lon"]))

else:
    st.markdown("### No vehicles yet — add one from the sidebar")

# -----------------------------
# AUTO REFRESH
# -----------------------------
time.sleep(1)
st.rerun()